In [1]:
import sys

print("Python Version:", sys.version)
print("Python Executable:", sys.executable)

Python Version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Python Executable: C:\Users\periy\anaconda3\python.exe


In [2]:
%pip install langchain langchain-community langchain-text-splitters langchain-chroma chromadb pypdf sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
import langchain
import chromadb
import pypdf
import sentence_transformers
print("LangChain:", langchain.__version__)
print("ChromaDB:", chromadb.__version__)
print("PDF library:", pypdf.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("RAG libraries installed successfully!")

LangChain: 1.4.4
ChromaDB: 1.5.9
PDF library: 6.20.0
Sentence Transformers: 6.1.0
RAG libraries installed successfully!


In [5]:
from pathlib import Path

project_dir = Path("DataScience-AI-RAG")
data_dir = project_dir / "data"

data_dir.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_dir.resolve())
print("Data folder:", data_dir.resolve())

Project folder: C:\Users\periy\DataScience-AI-RAG
Data folder: C:\Users\periy\DataScience-AI-RAG\data


In [6]:
from pathlib import Path
from pypdf import PdfReader

data_dir = Path("DataScience-AI-RAG/data")

pdf_files = list(data_dir.glob("*.pdf"))

print("PDF files found:", [file.name for file in pdf_files])

PDF files found: ['Machine Learning.pdf']


In [9]:
from pypdf import PdfReader

pdf_path = data_dir / "Machine Learning.pdf"

reader = PdfReader(str(pdf_path))

print("Total pages:", len(reader.pages))

for page_number, page in enumerate(reader.pages[:3], start=1):
    print(f"\n--- Page {page_number} ---")
    print(page.extract_text() or "No extractable text found")

Total pages: 3

--- Page 1 ---
Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care. 
• Need for Machine Learning  
• Machine Learning is needed for problems where traditional 
programming is difficult to scale or requires too many manually 
defined rules. It can learn patterns from data, automate tasks 
and make useful predictions.  
• 1. Solving Complex Problems  
• ML can handle tasks that are difficult to solve with fixed rules by 
learning patterns from data.  
Examples: 
• Image and speech recognition 
• Language translation and medical diag

In [10]:
from pypdf import PdfReader

pdf_path = data_dir / "Machine Learning.pdf"
reader = PdfReader(str(pdf_path))

documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        documents.append({
            "text": text,
            "page": page_number,
            "source": pdf_path.name
        })

print("Total documents:", len(documents))
print("First page:", documents[0]["page"])
print(documents[0]["text"][:500])

Total documents: 3
First page: 1
Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care.


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

pages = [
    Document(
        page_content=item["text"],
        metadata={
            "source": item["source"],
            "page": item["page"]
        }
    )
    for item in documents
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(pages)

print("Total pages:", len(pages))
print("Total chunks:", len(chunks))

Total pages: 3
Total chunks: 9


In [12]:
for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---")
    print("Page:", chunk.metadata["page"])
    print(chunk.page_content)


--- Chunk 1 ---
Page: 1
Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care. 
• Need for Machine Learning  
• Machine Learning is needed for problems where traditional 
programming is difficult to scale or requires too many manually

--- Chunk 2 ---
Page: 1
programming is difficult to scale or requires too many manually 
defined rules. It can learn patterns from data, automate tasks 
and make useful predictions.  
• 1. Solving Complex Problems  
• ML can handle tasks that are difficult to solve with fixed rules by 
learning patterns from data.  

In [13]:
%pip install langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [17]:
vectors = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks[:3]]
)

print("Number of vectors:", len(vectors))
print("Vector dimensions:", len(vectors[0]))

Number of vectors: 3
Vector dimensions: 384


In [18]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(project_dir / "chroma_db"),
    collection_name="machine_learning_notes"
)

print("Chunks stored in ChromaDB:", vector_store._collection.count())

Chunks stored in ChromaDB: 18


In [19]:
query = "What are the applications of Machine Learning?"

results = vector_store.similarity_search(
    query,
    k=3
)

print("Question:", query)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result.metadata["source"])
    print("Page:", result.metadata["page"])
    print("Content:", result.page_content)

Question: What are the applications of Machine Learning?

--- Result 1 ---
Source: Machine Learning.pdf
Page: 1
Content: Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care. 
• Need for Machine Learning  
• Machine Learning is needed for problems where traditional 
programming is difficult to scale or requires too many manually

--- Result 2 ---
Source: Machine Learning.pdf
Page: 1
Content: Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
expli

In [20]:
query = "What are the applications of Machine Learning?"

results = vector_store.similarity_search(query, k=3)

context = "\n\n".join(
    result.page_content for result in results
)

print("QUESTION:")
print(query)

print("\nRETRIEVED CONTEXT:")
print(context)

QUESTION:
What are the applications of Machine Learning?

RETRIEVED CONTEXT:
Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care. 
• Need for Machine Learning  
• Machine Learning is needed for problems where traditional 
programming is difficult to scale or requires too many manually

Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 


In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
You are a helpful Machine Learning tutor.

Answer the question using only the context provided below.
If the answer is not available in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
""")

print("RAG prompt created successfully!")

RAG prompt created successfully!


In [22]:
formatted_prompt = prompt_template.format(
    context=context,
    question=query
)

print(formatted_prompt)

Human: 
You are a helpful Machine Learning tutor.

Answer the question using only the context provided below.
If the answer is not available in the context, say:
"I could not find the answer in the provided document."

Context:
Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make predictions or decisions without being 
explicitly programmed for every task. 
• Helps identify patterns, automate tasks and improve predictions using 
historical data. 
• For example, an e-commerce business can analyse customer and sales 
data to understand buying behaviour, while a healthcare organisation 
can analyse patient data to identify health trends and improve patient 
care. 
• Need for Machine Learning  
• Machine Learning is needed for problems where traditional 
programming is difficult to scale or requires too many manually

Machine Learning (ML) is a branch of Artificial Intelligence that enables 
computers to learn from data and make p

In [23]:
!ollama --version

ollama version is 0.40.3


In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama run llama3.2:3b "Explain machine learning in simple words."

In [ ]:
%pip install langchain-ollama

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

print("Ollama connected successfully!")

In [ ]:
# Step 12: Generate an answer using RAG

query = "What are the applications of Machine Learning?"

# Retrieve relevant chunks from ChromaDB
results = vector_store.similarity_search(query, k=3)

# Combine retrieved text into context
context = "\n\n".join(
    result.page_content for result in results
)

# Prepare the prompt
formatted_prompt = prompt_template.format(
    context=context,
    question=query
)

# Generate an answer using Ollama
response = llm.invoke(formatted_prompt)

print("Question:", query)
print("\nRAG Answer:")
print(response.content)

In [ ]:
def ask_rag(question):
    # Step 1: Retrieve relevant chunks
    results = vector_store.similarity_search(question, k=3)

    # Step 2: Combine retrieved text
    context = "\n\n".join(
        result.page_content for result in results
    )

    # Step 3: Prepare the prompt
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # Step 4: Generate the answer
    response = llm.invoke(formatted_prompt)

    return response.content

In [ ]:
answer = ask_rag("What is Machine Learning?")
print(answer)

In [ ]:
answer = ask_rag("What are the challenges of Machine Learning?")
print(answer)